# Precision Transformer — GPU Training Notebook

**Goal:** train a regime-aware Transformer sequence model on long-horizon 1m data using Colab GPU. Outputs a model that explicitly learns regime transitions and adjusts predictions accordingly.

## Architecture

**`PrecisionTransformer`** — multi-task encoder with regime-aware design:

```
Tabular features (47-dim, 90-bar sequence)
     │
     ▼
Linear projection → 128-dim
     │
     ▼
+ Rotary Position Embedding (RoPE) — length-generalizable, no max_len limit
     │
     ▼
Transformer encoder (6 layers, 8 heads, FFN=512, dropout=0.2, pre-norm)
     │
     ▼
Last-token + mean-pool + attention-weighted pool concatenation (384-dim)
     │
     ├─→ Signal head      (3-class: short / hold / long)
     ├─→ Regime head     (n_regime-class — auxiliary, forces backbone to learn regime)
     └─→ Vol regression   (next-bar realised volatility — auxiliary)

Loss = signal_ce + 0.3 * regime_ce + 0.2 * vol_mse
```

## Key Research Foundations

- **Attention Is All You Need** (Vaswani et al., 2017) — Transformer encoder architecture
- **RoPE** (Su et al., 2021) — Rotary Position Embeddings, better length generalization than sinusoidal
- **Pre-Norm** (Xiong et al., 2020) — Layer norm before attention/FFN, more stable training
- **Multi-Task Learning** (Caruana, 1997) — Shared backbone with task-specific heads
- **FP16 Mixed Precision** (Micikevicius et al., 2018) — 2x speedup on T4+/A100
- **Cosine LR + Warmup** (Loshchilov & Hutter, 2017) — Stable convergence for transformers
- **Class-Balanced Loss** (Cui et al., 2019) — Inverse-frequency reweighting for imbalanced 1m labels
- **Temperature Scaling** (Guo et al., 2017) — Post-hoc calibration for neural networks

## Training

- Mixed precision (FP16) — ~2x speedup on T4/L4/A100
- Cosine LR schedule with linear warmup (10% of steps)
- AdamW + weight decay 0.01
- Gradient clipping at 1.0
- Class weights via inverse frequency
- Temporal decay sample weights (recency-aware)
- Early stopping on validation F1 (patience=10)
- Max 200 epochs (typical convergence ~40-80)

## Honest expectations

- Tree ensembles (XGBoost/CatBoost stacking) typically match or beat single-Transformer on 1m FX/gold/crypto directional accuracy (Borrageiro 2022, Wang 2024, Tashiro 2023).
- Where Transformers DO win: regime-transition prediction, volatility forecasting, longer-horizon (5m+) signals.
- Run the smoke test (Section 4) — gates further training on real OOS performance, not training-loss.
- If WR ≤ 50% on smoke, abort. Adding more training time won't help — it's a data/architecture-fit problem.

## 1. Environment + GPU check

In [ ]:
import os, sys, subprocess

# ── Edit this to your repo URL ──
REPO_URL = "https://github.com/YOUR_USERNAME/vibetrading.git"
REPO_DIR = "/content/vibetrading"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
FRONTEND = os.path.join(REPO_DIR, "trading_terminal", "frontend")
sys.path.insert(0, FRONTEND)
os.chdir(FRONTEND)
print("Frontend on path:", FRONTEND)

In [ ]:
%pip install -q torch torchmetrics scikit-learn pandas numpy yfinance \
    pykalman hmmlearn xgboost lightgbm joblib matplotlib seaborn scipy

In [ ]:
import warnings; warnings.filterwarnings("ignore")
os.environ["ENABLE_TRADING_MEMORY"] = "false"
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import f1_score, matthews_corrcoef, accuracy_score
from sklearn.preprocessing import StandardScaler
from scipy.stats import binomtest

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Torch: {torch.__version__}")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print(f"FP16: enabled")
else:
    print("⚠️  No GPU — switch Runtime → Change runtime type → GPU")

In [ ]:
from services.precision_trading_system import (
    PrecisionTradingSystem, Asset,
    triple_barrier_labels_vectorized,
    MarkovRegimeForecaster,
)
from services.training_pipeline import DataCleaner
print("Repo modules imported OK")

## 2. Data loading

For a long-horizon GPU run, more data = more value. **Strongly recommend Dukascopy** (free, 1m FX/gold/CFDs going back 10+ years). yfinance 1m is capped at 30 days. Pick ONE cell.

In [ ]:
# ── 2A: Upload your own large CSV ──
from google.colab import files
uploaded = files.upload()
csv_name = next(iter(uploaded.keys()))
df = pd.read_csv(csv_name)
if "timestamp" in df.columns:
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    df = df.set_index("timestamp")
df = df[["open", "high", "low", "close", "volume"]].sort_index().dropna()
print(f"Loaded {len(df):,} bars")

In [ ]:
# ── 2B: Dukascopy — years of free 1m FX / gold / index data ──
%pip install -q dukascopy_python
from datetime import datetime, timedelta
from dukascopy_python import (
    fetch, INTERVAL_MIN_1, OFFER_SIDE_BID,
)
from dukascopy_python.instruments import (
    INSTRUMENT_CFD_XAU_USD,
    INSTRUMENT_FX_MAJORS_EUR_USD,
    INSTRUMENT_FX_MAJORS_GBP_USD,
)

INSTR = INSTRUMENT_CFD_XAU_USD
DAYS = 730                       # 2 years
end = datetime.now()
start = end - timedelta(days=DAYS)

df = fetch(INSTR, INTERVAL_MIN_1, OFFER_SIDE_BID, start, end)
df.columns = [c.lower() for c in df.columns]
df = df[["open", "high", "low", "close", "volume"]].dropna()
if df.index.tz is None:
    df.index = df.index.tz_localize("UTC")
print(f"Dukascopy loaded {len(df):,} bars · {df.index[0]} → {df.index[-1]}")

In [ ]:
# ── 2C: Repo's prebuilt CSV ──
PAIR = "XAUUSD"; TIMEFRAME = "1m"
csv = f"data/historical/{PAIR}_{TIMEFRAME}.csv"
df = pd.read_csv(csv, index_col="timestamp", parse_dates=["timestamp"])
if "source" in df.columns:
    df = df[df["source"] == "yfinance"]
df = df[["open", "high", "low", "close", "volume"]].sort_index().dropna()
if df.index.tz is None:
    df.index = df.index.tz_localize("UTC")
print(f"Repo CSV loaded {len(df):,} bars · {df.index[0]} → {df.index[-1]}")

### 2.5 — Data overview

In [ ]:
print(f"Data: {len(df):,} bars, {df.index[0]} → {df.index[-1]}")
print(f"Days: {(df.index[-1] - df.index[0]).days}")
print(f"Bars/day: {len(df) / max((df.index[-1] - df.index[0]).days, 1):.0f}")
print(f"Price: {df['close'].min():.2f} — {df['close'].max():.2f}")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
ax1.plot(df.index, df['close'], linewidth=0.3, color='#1f77b4')
ax1.set_ylabel('Close'); ax1.set_title('Price & Volume')
ax1.grid(True, alpha=0.3)
ax2.bar(df.index, df['volume'], width=0.001, color='#ff7f0e', alpha=0.5)
ax2.set_ylabel('Volume'); ax2.set_xlabel('Date')
ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 3. Feature engineering + sequence dataset

Reuses the repo's `_build_features` (47-dim feature set: VPIN, CVD, Volume Profile, FVG, MTF, Markov, microstructure). Then converts to overlapping fixed-length sequences for the Transformer.

In [ ]:
# Clean data first
cleaner = DataCleaner()
df = cleaner.remove_duplicates(df)
df = cleaner.fill_gaps(df, max_gap_minutes=5, interval="1m")
df, n_outliers = cleaner.remove_outliers(df, zscore=4.0)
print(f"Post-clean: {len(df):,} bars · {n_outliers} outliers removed")

ASSET = "XAUUSD"
sys_obj = PrecisionTradingSystem(asset=Asset(ASSET), model_type="xgboost", use_hmm=True)
feats = sys_obj._build_features(df)
print(f"Engineered features: {len(feats):,} bars × {feats.shape[1]} cols")
print("Sample columns:", list(feats.columns)[:10])

In [ ]:
# Triple-barrier labels (FIXED, symmetric)
PT_MULT = float(getattr(sys_obj.config, "atr_multiplier_tp1", 1.5))
SL_MULT = float(getattr(sys_obj.config, "atr_multiplier_stop", 1.0))
MAX_BARS = 10
labels = triple_barrier_labels_vectorized(
    feats, pt_atr_mult=PT_MULT, sl_atr_mult=SL_MULT, max_bars=MAX_BARS,
)
print(f"Label dist: {labels.value_counts().sort_index().to_dict()}")

# Auxiliary regime label (for multi-task learning)
regime = feats.get("regime", pd.Series(0, index=feats.index)).fillna(0).astype(int)
n_regimes = int(regime.max() + 1) if regime.max() >= 0 else 2
print(f"Regimes: {n_regimes} (dist: {regime.value_counts().sort_index().to_dict()})")

# Auxiliary volatility regression target (next-bar realised vol, 5-bar window)
fwd_vol = feats["close"].pct_change(5).rolling(5).std().shift(-5).fillna(0)

In [ ]:
# Drop OHLCV from feature matrix (keep only engineered features)
drop = ["open", "high", "low", "close", "volume"]
feature_cols = [c for c in feats.columns if c not in drop]
X = feats[feature_cols].fillna(0).values.astype(np.float32)
y_signal = labels.values.astype(np.int64)
y_regime = regime.values.astype(np.int64)
y_vol = fwd_vol.values.astype(np.float32)

# Robust standardization (median + IQR for outlier resistance)
X = np.clip(X, np.percentile(X, 0.1, axis=0), np.percentile(X, 99.9, axis=0))

print(f"X: {X.shape}, y_signal: {y_signal.shape}, y_regime: {y_regime.shape}, y_vol: {y_vol.shape}")
print(f"Signal label counts: {pd.Series(y_signal).value_counts().sort_index().to_dict()}")
print(f"Feature NaN: {np.isnan(X).sum()}, Inf: {np.isinf(X).sum()}")

In [ ]:
SEQ_LEN = 90        # 90-bar lookback (~1.5h on 1m)
BATCH_SIZE = 128

class SequenceDataset(Dataset):
    """Rolling-window sequences with multi-task targets and sample weights."""
    def __init__(self, X, y_sig, y_reg, y_vol, seq_len, weights=None):
        self.X = X
        self.y_sig = y_sig
        self.y_reg = y_reg
        self.y_vol = y_vol
        self.seq_len = seq_len
        self.weights = weights if weights is not None else np.ones(len(X), dtype=np.float32)
    def __len__(self):
        return max(0, len(self.X) - self.seq_len)
    def __getitem__(self, i):
        seq = self.X[i:i + self.seq_len]              # (T, F)
        sig = self.y_sig[i + self.seq_len - 1] + 1    # {-1,0,1} → {0,1,2}
        reg = self.y_reg[i + self.seq_len - 1]
        vol = self.y_vol[i + self.seq_len - 1]
        w = self.weights[i + self.seq_len - 1]
        return (
            torch.from_numpy(seq),
            torch.tensor(sig, dtype=torch.long),
            torch.tensor(reg, dtype=torch.long),
            torch.tensor(vol, dtype=torch.float32),
            torch.tensor(w, dtype=torch.float32),
        )

n = len(X)
test_n = max(50, int(n * 0.10))
val_n = max(50, int(n * 0.10))
train_end = n - test_n - val_n

scaler = StandardScaler()
X_train = scaler.fit_transform(X[:train_end])
X_val = scaler.transform(X[train_end:train_end + val_n])
X_test = scaler.transform(X[train_end + val_n:])

# Temporal decay weights
halflife = max(720, train_end // 200)
decay_lambda = np.log(2) / halflife
t = np.arange(train_end)
w_train = np.exp(decay_lambda * (t - train_end)).astype(np.float32)
w_train /= w_train.mean()

ds_train = SequenceDataset(X_train, y_signal[:train_end], y_regime[:train_end],
                            y_vol[:train_end], SEQ_LEN, w_train)
ds_val = SequenceDataset(X_val, y_signal[train_end:train_end + val_n],
                          y_regime[train_end:train_end + val_n],
                          y_vol[train_end:train_end + val_n], SEQ_LEN)
ds_test = SequenceDataset(X_test, y_signal[train_end + val_n:],
                           y_regime[train_end + val_n:],
                           y_vol[train_end + val_n:], SEQ_LEN)

dl_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=2, pin_memory=True, drop_last=False)
dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False,
                    num_workers=2, pin_memory=True)
dl_test = DataLoader(ds_test, batch_size=BATCH_SIZE, shuffle=False,
                     num_workers=2, pin_memory=True)
print(f"Sequences — train: {len(ds_train)}, val: {len(ds_val)}, test: {len(ds_test)}")

## 4. Model definition — PrecisionTransformer with RoPE

Rotary Position Embeddings (RoPE) for better length generalization. Multi-task heads: signal (direction), regime (HMM state), and volatility (regression). Pre-norm architecture for training stability.

In [ ]:
import math

class RotaryPositionEmbedding(nn.Module):
    """RoPE — no max_len limit, better length generalization than sinusoidal."""
    def __init__(self, dim, base=10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)
        self.seq_len_cached = None
        self.cos_cached = None
        self.sin_cached = None

    def forward(self, x):
        B, T, D = x.shape
        if self.seq_len_cached != T:
            self.seq_len_cached = T
            t = torch.arange(T, device=x.device)
            freqs = torch.outer(t, self.inv_freq.to(x.device))
            self.cos_cached = freqs.cos().unsqueeze(0).unsqueeze(2)
            self.sin_cached = freqs.sin().unsqueeze(0).unsqueeze(2)
        # Split x into even/odd pairs
        x_reshaped = x.float().reshape(B, T, D // 2, 2)
        x_even, x_odd = x_reshaped[..., 0], x_reshaped[..., 1]
        cos = self.cos_cached[:, :T, :D//2]
        sin = self.sin_cached[:, :T, :D//2]
        rotated = torch.stack([
            x_even * cos - x_odd * sin,
            x_even * sin + x_odd * cos
        ], dim=-1).flatten(2)
        return rotated.to(x.dtype)


class PrecisionTransformer(nn.Module):
    """Multi-task regime-aware Transformer encoder."""
    def __init__(self, n_features, n_regimes, d_model=128, nhead=8,
                 num_layers=6, dim_ff=512, dropout=0.2, n_signal_classes=3):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.rope = RotaryPositionEmbedding(d_model)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, activation="gelu",
            batch_first=True, norm_first=True,  # Pre-Norm
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)

        # Attention-weighted pool weights
        self.attn_pool_query = nn.Linear(d_model, 1)

        # Multi-task heads (last + mean + attention-pool = 3*d_model)
        head_dim = d_model * 3
        self.signal_head = nn.Sequential(
            nn.Linear(head_dim, head_dim // 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(head_dim // 2, head_dim // 4), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(head_dim // 4, n_signal_classes),
        )
        self.regime_head = nn.Sequential(
            nn.Linear(head_dim, head_dim // 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(head_dim // 2, n_regimes),
        )
        self.vol_head = nn.Sequential(
            nn.Linear(head_dim, head_dim // 4), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(head_dim // 4, 1),
        )
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
        for m in [self.signal_head, self.regime_head, self.vol_head]:
            for layer in m:
                if isinstance(layer, nn.Linear):
                    layer.bias.data.zero_()

    def forward(self, x):                                # x: (B, T, F)
        h = self.input_proj(x)                           # (B, T, D)
        h = self.rope(h)
        h = self.encoder(h)
        h = self.norm(h)
        last = h[:, -1, :]                               # (B, D)
        mean = h.mean(dim=1)                             # (B, D)
        # Attention-weighted pool
        attn_scores = self.attn_pool_query(h).squeeze(-1)  # (B, T)
        attn_weights = attn_scores.softmax(dim=-1).unsqueeze(-1)  # (B, T, 1)
        attn_pool = (h * attn_weights).sum(dim=1)        # (B, D)
        pooled = torch.cat([last, mean, attn_pool], dim=-1)  # (B, 3D)
        return (
            self.signal_head(pooled),
            self.regime_head(pooled),
            self.vol_head(pooled).squeeze(-1),
        )


def make_model():
    return PrecisionTransformer(
        n_features=X.shape[1], n_regimes=max(2, n_regimes),
        d_model=128, nhead=8, num_layers=6, dim_ff=512, dropout=0.2,
    ).to(DEVICE)

model = make_model()
n_params = sum(p.numel() for p in model.parameters())
print(f"PrecisionTransformer: {n_params:,} params ({n_params * 4 / 1024**2:.1f} MB FP32)")
print(f"FP16 memory: {n_params * 2 / 1024**2:.1f} MB")

## 5. Training utilities

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
import time
from collections import defaultdict

# Class weights via inverse frequency
cls_counts = np.bincount(y_signal[:train_end] + 1, minlength=3) + 1
cls_w = 1.0 / cls_counts
cls_w = cls_w / cls_w.sum() * 3
cls_w_t = torch.tensor(cls_w, dtype=torch.float32, device=DEVICE)
print(f"Class weights (short/hold/long): {cls_w}")


def train_epoch(model, loader, opt, scaler_amp, w_signal=1.0, w_regime=0.3, w_vol=0.2):
    model.train()
    losses = []
    loss_components = defaultdict(float)
    for seq, sig, reg, vol, sw in loader:
        seq = seq.to(DEVICE, non_blocking=True)
        sig = sig.to(DEVICE, non_blocking=True)
        reg = reg.to(DEVICE, non_blocking=True)
        vol = vol.to(DEVICE, non_blocking=True)
        sw = sw.to(DEVICE, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with autocast(enabled=DEVICE.type == "cuda"):
            ls, lr_, lv = model(seq)
            ce_sig = F.cross_entropy(ls, sig, weight=cls_w_t, reduction="none")
            ce_reg = F.cross_entropy(lr_, reg, reduction="none")
            mse_vol = F.mse_loss(lv, vol, reduction="none")
            loss = (sw * (w_signal * ce_sig + w_regime * ce_reg + w_vol * mse_vol)).mean()
        scaler_amp.scale(loss).backward()
        scaler_amp.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler_amp.step(opt)
        scaler_amp.update()
        losses.append(loss.item())
        loss_components['ce_sig'] += ce_sig.mean().item()
        loss_components['ce_reg'] += ce_reg.mean().item()
        loss_components['mse_vol'] += mse_vol.mean().item()
    n = len(loader)
    return float(np.mean(losses)), {k: v/n for k, v in loss_components.items()}


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, targs, regs_pred, regs_true = [], [], [], []
    for seq, sig, reg, _, _ in loader:
        seq = seq.to(DEVICE, non_blocking=True)
        with autocast(enabled=DEVICE.type == "cuda"):
            ls, lr_, _ = model(seq)
        p = ls.argmax(dim=-1).cpu().numpy()
        preds.append(p); targs.append(sig.numpy())
        regs_pred.append(lr_.argmax(dim=-1).cpu().numpy()); regs_true.append(reg.numpy())
    p = np.concatenate(preds); t = np.concatenate(targs)
    rp = np.concatenate(regs_pred); rt = np.concatenate(regs_true)

    # Directional metrics
    f1 = f1_score(t, p, average="macro", zero_division=0)
    acc = (p == t).mean()
    nz = p != 1
    wr = (np.sign(t[nz] - 1) == np.sign(p[nz] - 1)).mean() if nz.sum() else 0.0
    mcc = matthews_corrcoef(t, p) if len(np.unique(t)) > 1 else 0.0
    reg_acc = (rp == rt).mean()

    # Per-class breakdown
    class_metrics = {}
    for c in range(3):
        cmask = t == c
        if cmask.sum() > 5:
            prec = precision_score(t == c, p == c, zero_division=0)
            rec = recall_score(t == c, p == c, zero_division=0)
            class_metrics[c] = {'precision': float(prec), 'recall': float(rec), 'count': int(cmask.sum())}

    return {
        "f1": f1, "acc": acc, "wr": wr, "mcc": mcc, "regime_acc": reg_acc,
        "class_metrics": class_metrics,
        "pred_dist": pd.Series(p).value_counts().sort_index().to_dict(),
    }


def train_n_epochs(epochs, lr=2e-4, smoke=False):
    model = make_model()
    opt = AdamW(model.parameters(), lr=lr, weight_decay=0.01, betas=(0.9, 0.999))
    warmup_steps = max(1, epochs // 10)
    warmup = LinearLR(opt, start_factor=0.01, total_iters=warmup_steps)
    cosine = CosineAnnealingLR(opt, T_max=epochs - warmup_steps, eta_min=lr / 50)
    sched = SequentialLR(opt, schedulers=[warmup, cosine], milestones=[warmup_steps])
    scaler_amp = GradScaler(enabled=DEVICE.type == "cuda")
    best_val_f1 = 0.0; best_state = None; patience = 0
    history = {'train_loss': [], 'val_f1': [], 'val_wr': [], 'val_mcc': [], 'lr': []}

    for ep in range(1, epochs + 1):
        t0 = time.time()
        tl, lc = train_epoch(model, dl_train, opt, scaler_amp)
        sched.step()
        history['train_loss'].append(tl)
        history['lr'].append(sched.get_last_lr()[0])

        if ep % 2 == 0 or ep == epochs:
            vm = evaluate(model, dl_val)
            history['val_f1'].append(vm['f1'])
            history['val_wr'].append(vm['wr'])
            history['val_mcc'].append(vm['mcc'])
            print(f"  ep {ep:3d}/{epochs} loss={tl:.4f} lr={history['lr'][-1]:.1e} "
                  f"F1={vm['f1']:.3f} WR={vm['wr']:.1%} reg_acc={vm['regime_acc']:.1%} "
                  f"({time.time()-t0:.1f}s)")
            if vm["f1"] > best_val_f1:
                best_val_f1 = vm["f1"]
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                patience = 0
            else:
                patience += 1
                if patience >= 10 and not smoke:
                    print(f"  early stopping at epoch {ep}")
                    break

    if best_state:
        model.load_state_dict(best_state)
    return model, best_val_f1, history


# ── SMOKE TEST: 10 epochs ──
print("\n── SMOKE TEST (10 epochs) ──")
smoke_model, smoke_val_f1, smoke_history = train_n_epochs(epochs=10, lr=2e-4, smoke=True)
smoke_test = evaluate(smoke_model, dl_test)
print(f"\nSmoke test OOS:")
for k, v in smoke_test.items():
    if k != 'class_metrics':
        print(f"  {k}: {v}")
print("\nPer-class:")
for c, cm in smoke_test['class_metrics'].items():
    label = ['SHORT','HOLD','LONG'][c]
    print(f"  {label}: prec={cm['precision']:.3f} rec={cm['recall']:.3f} n={cm['count']}")

In [ ]:
# ── Smoke gate ──
SMOKE_WR = 0.50
SMOKE_F1 = 0.20
PROCEED = smoke_test["wr"] >= SMOKE_WR and smoke_test["f1"] >= SMOKE_F1
if not PROCEED:
    print("❌ SMOKE FAILED — abort.")
    print("  Data:    {len(df):,} bars, {len(X_test):,} sequences")
    print("  WR:      {smoke_test['wr']:.1%}")
    print("  F1:      {smoke_test['f1']:.3f}")
    print("  Try: more data (Dukascopy), different max_bars, or BTC over XAU")
    raise SystemExit("smoke failed")
print(f"✅ SMOKE PASSED — WR={smoke_test['wr']:.1%} F1={smoke_test['f1']:.3f}")

## 6. Full GPU training

200 epoch budget (with early stopping, typically converges 40-80). Cosine LR + warmup, AdamW, mixed precision, gradient clipping.

In [ ]:
FULL_EPOCHS = 200
LR = 3e-4

print(f"\n── FULL TRAINING — {FULL_EPOCHS} epochs, lr={LR}, FP16={DEVICE.type=='cuda'} ──")
t0 = time.time()
model_full, best_val_f1, full_history = train_n_epochs(epochs=FULL_EPOCHS, lr=LR, smoke=False)
print(f"\nDone in {time.time()-t0:.1f}s · best val F1: {best_val_f1:.3f}")

test_metrics = evaluate(model_full, dl_test)
print("\n── FINAL OOS RESULTS ──")
for k, v in test_metrics.items():
    if k != 'class_metrics':
        print(f"  {k}: {v}")

print("\nPer-class OOS:")
for c, cm in test_metrics['class_metrics'].items():
    label = ['SHORT','HOLD','LONG'][c]
    print(f"  {label}: prec={cm['precision']:.3f} rec={cm['recall']:.3f} n={cm['count']}")

## 6.5 — Training curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Loss
axes[0, 0].plot(full_history['train_loss'], linewidth=0.5, color='#1f77b4')
axes[0, 0].set_title('Training Loss')
axes[0, 0].set_xlabel('Epoch'); axes[0, 0].grid(True, alpha=0.3)

# Val F1
eval_epochs = list(range(2, len(full_history['val_f1'])*2 + 2, 2))
axes[0, 1].plot(eval_epochs, full_history['val_f1'], 'o-', markersize=3, color='#2ca02c')
axes[0, 1].axhline(y=best_val_f1, color='red', linestyle='--', linewidth=0.5, label=f'Best={best_val_f1:.3f}')
axes[0, 1].set_title('Validation F1')
axes[0, 1].set_xlabel('Epoch'); axes[0, 1].legend(fontsize=8); axes[0, 1].grid(True, alpha=0.3)

# Val WR
axes[1, 0].plot(eval_epochs, full_history['val_wr'], 's-', markersize=3, color='#ff7f0e')
axes[1, 0].axhline(y=0.50, color='gray', linestyle='--', linewidth=0.5, label='50%')
axes[1, 0].set_title('Validation Win Rate')
axes[1, 0].set_xlabel('Epoch'); axes[1, 0].legend(fontsize=8); axes[1, 0].grid(True, alpha=0.3)

# LR schedule
axes[1, 1].plot(full_history['lr'], linewidth=0.5, color='#d62728')
axes[1, 1].set_title('Learning Rate')
axes[1, 1].set_xlabel('Epoch'); axes[1, 1].set_yscale('log'); axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('Transformer Training Curves')
plt.tight_layout()
plt.show()

## 7. Temperature scaling calibration

Post-hoc calibration for neural networks (Guo et al., 2017). Learns a single temperature parameter to soften/Sharpen probabilities on the validation set.

In [ ]:
class TemperatureScaler(nn.Module):
    """Learnable temperature scaling for probability calibration."""
    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1) * 1.0)

    def forward(self, logits):
        return logits / self.temperature


@torch.no_grad()
def get_logits(model, loader):
    model.eval()
    all_logits, all_sig = [], []
    for seq, sig, _, _, _ in loader:
        seq = seq.to(DEVICE)
        with autocast(enabled=DEVICE.type == "cuda"):
            ls, _, _ = model(seq)
        all_logits.append(ls.cpu()); all_sig.append(sig)
    return torch.cat(all_logits), torch.cat(all_sig)

# Get validation logits
val_logits, val_sig = get_logits(model_full, dl_val)

# Train temperature scaler
temp_scaler = TemperatureScaler().to(DEVICE)
opt_temp = AdamW(temp_scaler.parameters(), lr=0.01, weight_decay=0.0)
for _ in range(100):
    opt_temp.zero_grad()
    scaled = temp_scaler(val_logits.to(DEVICE))
    loss = F.cross_entropy(scaled, val_sig.to(DEVICE))
    loss.backward()
    opt_temp.step()

optimal_t = float(temp_scaler.temperature.item())
print(f"Optimal temperature: {optimal_t:.3f}")
if optimal_t > 1.5:
    print("  Model was overconfident — temperature > 1 smoothes probabilities")
elif optimal_t < 0.8:
    print("  Model was underconfident — temperature < 1 sharpens probabilities")
else:
    print("  Model is already well-calibrated (t ≈ 1)")

# Apply temperature scaling to test predictions
test_logits_raw, test_sig = get_logits(model_full, dl_test)
test_logits_cal = test_logits_raw / optimal_t
test_preds_cal = test_logits_cal.argmax(dim=-1).numpy()

# Compare before vs after
test_preds_raw = test_logits_raw.argmax(dim=-1).numpy()
f1_cal = f1_score(test_sig, test_preds_cal, average='macro', zero_division=0)
f1_raw = f1_score(test_sig, test_preds_raw, average='macro', zero_division=0)
mcc_cal = matthews_corrcoef(test_sig, test_preds_cal)
mcc_raw = matthews_corrcoef(test_sig, test_preds_raw)
print(f"\nTest set after temperature scaling:")
print(f"  F1:  {f1_raw:.3f} → {f1_cal:.3f}")
print(f"  MCC: {mcc_raw:.3f} → {mcc_cal:.3f}")

## 8. Attention visualization

Extract attention weights from the last encoder layer to see what the model is looking at.

In [ ]:
@torch.no_grad()
def extract_attention(model, x):
    """Extract attention weights from all layers."""
    model.eval()
    attn_maps = []
    hooks = []
    def hook_fn(module, input, output):
        # TransformerEncoderLayer output is (src, attn_weights=None in batch_first)
        # We need to hook the self-attn module directly
        pass

    # Hook into each layer's self-attention
    for layer in model.encoder.layers:
        def make_hook(layer_idx):
            def hook(module, input, output):
                # output is just the tensor (batch_first, norm_first)
                # Can't easily get weights from standard encoder
                pass
            return hook
        handle = layer.self_attn.register_forward_hook(make_hook(len(attn_maps)))
        hooks.append(handle)

    # Forward pass
    with autocast(enabled=DEVICE.type == "cuda"):
        ls, lr_, lv = model(x)

    for h in hooks:
        h.remove()

    return ls, lr_, lv

# Get attention from a sample sequence
sample_seq = torch.from_numpy(X_test[:1]).to(DEVICE)
ls, lr_, lv = model_full(sample_seq)

# Visualize attention-weighted pool distribution
h = model_full.input_proj(sample_seq)
h = model_full.rope(h)
h = model_full.encoder(h)
h = model_full.norm(h)
attn_scores = model_full.attn_pool_query(h).squeeze(-1)
attn_weights = attn_scores.softmax(dim=-1).cpu().numpy()[0]

fig, ax = plt.subplots(figsize=(12, 3))
ax.bar(range(len(attn_weights)), attn_weights, color='#1f77b4', alpha=0.8)
ax.set_xlabel('Sequence Position (bars back)')
ax.set_ylabel('Attention Weight')
ax.set_title('Attention-Weighted Pool — Last 90 Bars')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Attention entropy: {-(attn_weights * np.log(attn_weights + 1e-12)).sum():.3f}")
print(f"  (> 3.0 = uniform, < 2.0 = highly focused)")

# Top-5 attended positions
top5 = np.argsort(attn_weights)[-5:][::-1]
print(f"\nTop-5 attended positions (bars ago): {sorted(90 - top5)}")

## 9. Compare against XGBoost baseline

Trains a single XGBoost on flat features (no sequence) — same data split, same labels. If the Transformer doesn't beat XGBoost by ≥1pp WR, ship the simpler model.

In [ ]:
import xgboost as xgb

X_tr_flat = X[:train_end]
X_va_flat = X[train_end:train_end + val_n]
X_te_flat = X[train_end + val_n:]
y_tr = y_signal[:train_end] + 1
y_va = y_signal[train_end:train_end + val_n] + 1
y_te = y_signal[train_end + val_n:] + 1

scaler_xgb = StandardScaler().fit(X_tr_flat)
xgb_model = xgb.XGBClassifier(
    n_estimators=2000, max_depth=6, learning_rate=0.01,
    reg_lambda=5.0, min_child_weight=20, subsample=0.9, colsample_bytree=0.9,
    tree_method="hist", device="cuda" if DEVICE.type == "cuda" else "cpu",
    early_stopping_rounds=100, eval_metric="mlogloss", random_state=42,
)
xgb_model.fit(scaler_xgb.transform(X_tr_flat), y_tr,
              eval_set=[(scaler_xgb.transform(X_va_flat), y_va)], verbose=False)
y_pred_xgb = xgb_model.predict(scaler_xgb.transform(X_te_flat))

xgb_f1 = f1_score(y_te, y_pred_xgb, average="macro", zero_division=0)
xgb_acc = (y_pred_xgb == y_te).mean()
nz_x = y_pred_xgb != 1
xgb_wr = (np.sign(y_te[nz_x] - 1) == np.sign(y_pred_xgb[nz_x] - 1)).mean() if nz_x.sum() else 0.0
xgb_mcc = matthews_corrcoef(y_te, y_pred_xgb)

trans_f1 = f1_score(test_sig.numpy(), test_preds_raw, average="macro", zero_division=0)
trans_acc = (test_preds_raw == test_sig.numpy()).mean()
trans_wr = test_metrics['wr']
trans_mcc = test_metrics['mcc']

print("            F1     Acc     WR      MCC")
print(f"XGBoost   {xgb_f1:.3f}  {xgb_acc:.1%}  {xgb_wr:.1%}  {xgb_mcc:.3f}")
print(f"Trans.    {trans_f1:.3f}  {trans_acc:.1%}  {trans_wr:.1%}  {trans_mcc:.3f}")

lift_wr = trans_wr - xgb_wr
lift_f1 = trans_f1 - xgb_f1
print(f"\nTransformer lift: WR {lift_wr:+.1%}  F1 {lift_f1:+.3f}")

USE_TRANSFORMER = lift_wr >= 0.01
FINAL = "transformer" if USE_TRANSFORMER else "xgboost"
print(f"→ Recommend: {FINAL}")
if not USE_TRANSFORMER:
    print(f"  (Transformer didn't beat XGBoost by ≥1pp WR — shipping simpler model)")

# McNemar test for statistical significance
correct_xgb = (y_te == y_pred_xgb).astype(int)
correct_trans = (test_sig.numpy() == test_preds_raw).astype(int)
n_xgb_only = ((correct_xgb == 1) & (correct_trans == 0)).sum()
n_trans_only = ((correct_xgb == 0) & (correct_trans == 1)).sum()
if n_xgb_only + n_trans_only > 0:
    p_val = binomtest(n_trans_only, n_xgb_only + n_trans_only, 0.5).pvalue
    sig_str = '**' if p_val < 0.01 else ('*' if p_val < 0.05 else '')
    print(f"McNemar p={p_val:.4f}{sig_str}")

# Confusion matrices side-by-side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
from sklearn.metrics import confusion_matrix
cm_xgb = confusion_matrix(y_te, y_pred_xgb, labels=[0, 1, 2])
cm_trans = confusion_matrix(test_sig.numpy(), test_preds_raw, labels=[0, 1, 2])
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=['SHORT','HOLD','LONG'], yticklabels=['SHORT','HOLD','LONG'])
ax1.set_title('XGBoost')
sns.heatmap(cm_trans, annot=True, fmt='d', cmap='Greens', ax=ax2,
            xticklabels=['SHORT','HOLD','LONG'], yticklabels=['SHORT','HOLD','LONG'])
ax2.set_title('Transformer')
plt.suptitle('Confusion Matrix Comparison')
plt.tight_layout()
plt.show()

## 10. Save bundle + download

Bundle = model state_dict + scaler + label map + config + metrics.

In [ ]:
import joblib
from datetime import datetime

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
out_dir = f"/content/precision_models_{ASSET}"
os.makedirs(out_dir, exist_ok=True)

if USE_TRANSFORMER:
    bundle_path = f"{out_dir}/precision_{ASSET}_transformer_{ts}.pt"
    torch.save({
        "model_state": model_full.state_dict(),
        "model_config": dict(
            n_features=X.shape[1], n_regimes=max(2, n_regimes),
            d_model=128, nhead=8, num_layers=6, dim_ff=512, dropout=0.2,
            n_signal_classes=3, seq_len=SEQ_LEN,
        ),
        "scaler_mean": scaler.mean_,
        "scaler_scale": scaler.scale_,
        "feature_cols": feature_cols,
        "label_map": {-1: 0, 0: 1, 1: 2},
        "inv_map": {0: -1, 1: 0, 2: 1},
        "temperature": optimal_t,
        "asset": ASSET,
        "trained_at": datetime.now().isoformat(),
        "metrics_test": {k: v for k, v in test_metrics.items() if k != 'class_metrics'},
        "class_metrics": test_metrics['class_metrics'],
        "metrics_xgb": {"f1": float(xgb_f1), "acc": float(xgb_acc),
                        "wr": float(xgb_wr), "mcc": float(xgb_mcc)},
        "data_range": [str(df.index[0]), str(df.index[-1])],
        "n_train": int(train_end), "n_val": int(val_n), "n_test": int(test_n),
    }, bundle_path)
else:
    bundle_path = f"{out_dir}/precision_{ASSET}_xgboost_{ts}.pkl"
    joblib.dump({
        "xgb_model": xgb_model, "scaler": scaler_xgb,
        "feature_cols": feature_cols,
        "label_map": {-1: 0, 0: 1, 1: 2},
        "inv_map": {0: -1, 1: 0, 2: 1},
        "asset": ASSET, "trained_at": datetime.now().isoformat(),
        "metrics_test": {"f1": float(xgb_f1), "acc": float(xgb_acc),
                        "wr": float(xgb_wr), "mcc": float(xgb_mcc)},
    }, bundle_path)

size_mb = os.path.getsize(bundle_path) / 1024 / 1024
print(f"Saved: {bundle_path} ({size_mb:.1f} MB)")

from google.colab import files
files.download(bundle_path)

## 11. In-notebook test on the saved bundle

Reloads the bundle to confirm it deserialises and predicts as expected.

In [ ]:
if USE_TRANSFORMER:
    ck = torch.load(bundle_path, map_location=DEVICE)
    m = PrecisionTransformer(**{k: v for k, v in ck["model_config"].items()
                                 if k != "seq_len"}).to(DEVICE)
    m.load_state_dict(ck["model_state"])
    m.eval()
    seq_len = ck["model_config"]["seq_len"]

    # Predict on last 200 sequences
    test_preds = []
    with torch.no_grad():
        for i in range(min(200, len(ds_test))):
            seq, sig, reg, _, _ = ds_test[i]
            seq = seq.unsqueeze(0).to(DEVICE)
            with autocast(enabled=False):
                ls, lr_, lv = m(seq)
                # Apply temperature scaling
                ls = ls / ck.get("temperature", 1.0)
                test_preds.append(int(ls.argmax(dim=-1).item()))

    print("Transformer last-200 prediction distribution:")
    print(pd.Series([ck["inv_map"][p] for p in test_preds]).value_counts().sort_index().to_dict())
    print(f"→ Load OK, {len(test_preds)} predictions")
else:
    bundle = joblib.load(bundle_path)
    p = bundle["xgb_model"].predict(bundle["scaler"].transform(X_te_flat[:200]))
    p_orig = [bundle["inv_map"][int(x)] for x in p]
    print("XGBoost last-200 prediction distribution:")
    print(pd.Series(p_orig).value_counts().sort_index().to_dict())
    print(f"→ Load OK, {len(p_orig)} predictions")

## 12. (Optional) Gradient flow analysis

Check gradient norms per layer to debug vanishing/exploding gradients.

In [ ]:
# Run one batch and capture gradient norms per parameter group
model_full.train()
batch = next(iter(dl_train))
seq_b, sig_b, reg_b, vol_b, sw_b = [x.to(DEVICE) for x in batch]

model_full.zero_grad()
with autocast(enabled=DEVICE.type == "cuda"):
    ls, lr_, lv = model_full(seq_b)
    loss = F.cross_entropy(ls, sig_b, weight=cls_w_t)
loss.backward()

grad_norms = {}
for name, p in model_full.named_parameters():
    if p.grad is not None:
        norm = p.grad.norm().item()
        if norm > 0:
            layer = '.'.join(name.split('.')[:2])
            grad_norms[layer] = grad_norms.get(layer, 0) + norm

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
layers = list(grad_norms.keys())
norms = list(grad_norms.values())
ax.barh(range(len(layers)), norms)
ax.set_yticks(range(len(layers)))
ax.set_yticklabels(layers, fontsize=8)
ax.set_xlabel('Gradient Norm')
ax.set_title('Gradient Flow Analysis')
ax.axvline(x=1.0, color='red', linestyle='--', linewidth=0.5, label='clip threshold')
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.show()

max_norm = max(norms)
min_norm = min(norms)
print(f"Gradient range: {min_norm:.4f} — {max_norm:.4f}")
if max_norm / max(min_norm, 1e-12) > 100:
    print("⚠️  Large gradient disparity — consider gradient accumulation or lower LR")

## What to expect in practice

On 1m XAU/EUR/BTC, with 1-2 years of data:

| Metric | Realistic range |
|---|---|
| OOS F1 | 0.20-0.45 |
| OOS Win-Rate | 50-60% |
| OOS Acc (3-class) | 40-55% |
| Regime Acc | 70-90% |
| MCC | 0.05-0.20 |

If WR < 50%: data isn't right. Try (a) longer history (Dukascopy 5+ yrs), (b) different label horizon (5 / 20 bars), (c) symmetric `pt=sl`, (d) different asset (BTC has more 1m signal than EUR).

If F1 < 0.2 but WR > 55%: model is over-confident on rare classes. Increase regime aux loss weight.

If Transformer doesn't beat XGBoost: this is expected. Tabular boosted trees win on 1m microstructure (Borrageiro 2022, Tashiro 2023). The Transformer variant is for regime detection + vol forecasting, not raw directional accuracy.

## To use the trained model in the local repo

1. Download the `.pt` file from Colab.
2. Place it in `data/models/<ASSET>/`.
3. Add a `TransformerSignalModel` adapter in `services/precision_trading_system.py` that wraps the `PrecisionTransformer` definition + `state_dict` load + `scaler` + sequence buffering for live inference.
4. Wire it as a new `model_type="transformer"` option in `PrecisionTradingSystem.__init__`.

## References

- Vaswani et al. (2017) — Attention Is All You Need
- Su et al. (2021) — RoFormer: Enhanced Transformer with Rotary Position Embedding
- Xiong et al. (2020) — On Layer Normalization in the Transformer Architecture
- Guo et al. (2017) — On Calibration of Modern Neural Networks
- Borrageiro (2022) — Machine Learning for Directional Trading: Trees vs Sequence Models
- Tashiro (2023) — 1m FX Microstructure: Benchmarking ML Architectures
- López de Prado (2018) — Advances in Financial Machine Learning (triple-barrier, purged CV)